In [0]:
pip install jinja2

In [0]:
from jinja2 import Template

In [0]:
parameters = [
    {
        "table": "databricksazurespotify.silver.fact_stream",
        "alias": "fact_stream",
        "cols": "fact_stream.stream_id, fact_stream.listen_duration"
    },
    {
        "table": "databricksazurespotify.silver.dim_user",
        "alias": "dim_user",
        "cols": "dim_user.user_id, dim_user.user_name",
        "condition": "fact_stream.user_id = dim_user.user_id"
    },
    {
        "table": "databricksazurespotify.silver.dim_track",
        "alias": "dim_track",
        "cols": "dim_track.artist_id, dim_track.track_name",
        "condition": "fact_stream.track_id = dim_track.track_id"
    }
]

In [0]:
query_txt = """
SELECT
    {% for param in parameters %}
        {{ param.cols }}
        {% if not loop.last %},{% endif %}
    {% endfor %}

FROM
    {{ parameters[0].table }} AS {{ parameters[0].alias }}

    {% for param in parameters[1:] %}
    LEFT JOIN {{ param.table }} AS {{ param.alias }}
        ON {{ param.condition }}
    {% endfor %}
"""

In [0]:
jinja_sql_str=Template(query_txt)
query=jinja_sql_str.render(parameters=parameters)
print(query)

In [0]:
display(spark.sql(query))